# End-to-End SLC Model Training Pipeline (Colab-Compatible)

This notebook gives one readable path from validated data loading through aggregate EDA, participant-safe splitting, fold-fitted preprocessing, three-domain model comparison, probability calibration, development-only threshold selection, locked-test evaluation, explainability, and model export.

**Research boundary:** this is a non-diagnostic early-childhood developmental monitoring experiment. It predicts questionnaire-defined follow-up concern outcomes; it does not diagnose developmental delay, autism, or any medical condition. No model in this repository has prospective, external, or clinical validation.

**Privacy boundary:** the notebook never displays participant-level rows or identifiers. Upload the processed CSV and split manifest only to your private Colab runtime. Do not publish the runtime, outputs, CSV, or manifest.

**Reproducibility boundary:** the repository's existing locked test has already been evaluated. Re-running this notebook is educational/reproducibility work, not a fresh independent validation. Do not tune anything after viewing locked-test results and then claim that the same test remains untouched.

## 1. Runtime setup

The versions below match the core repository environment. Restart the Colab runtime if installation replaces already-imported packages.

In [ ]:
import sys, subprocess
packages = [
    'numpy==2.2.2', 'scipy==1.15.1', 'pandas==2.2.3', 'scikit-learn==1.6.1',
    'matplotlib==3.10.0', 'joblib==1.4.2', 'xgboost==3.2.0'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Dependencies installed. In Colab, restart the runtime if requested, then continue.')

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import version
import json, math, warnings, zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

SEED = 20260822
N_SPLITS, N_REPEATS = 5, 5
EXPECTED_DATASET_SHA256 = '464A0CD22DE10A425AAA2921D862DBC040584EB8CC51C27140B3AB5946C419B3'
EXPECTED_MANIFEST_SHA256 = '97E62CFD4F2DDBABD59778D1D0508C3B4EDB77E87311FB3F03C31D95AFBC23A5'
CHECKPOINTS = ['CP2M','CP4M','CP6M','CP9M','CP12M','CP15M','CP18M','CP24M','CP30M','CP36M','CP48M','CP60M']
QUESTION_COLUMNS = [f'{d}_q{i}' for d in ['communication','cognitive','social_emotional'] for i in range(1,6)]
FEATURE_COLUMNS = ['baseline_checkpoint_id', *QUESTION_COLUMNS]
TARGET_COLUMNS = ['target_communication_concern','target_cognitive_concern','target_social_emotional_concern']
DOMAIN_NAMES = ['Communication', 'Cognitive', 'Social-emotional']
print(f'Python {sys.version.split()[0]} | pandas {pd.__version__} | scikit-learn {version("scikit-learn")}')

## 2. Load the private dataset and frozen split

For local execution from the repository, the paths are discovered automatically. In Colab, run the cell and upload these two private files when prompted:

- `survey-ml-cross-checkpoint-classification.csv`
- `split_manifest.csv`

The split manifest contains pseudonymous participant identifiers. Keep it private. The notebook deliberately refuses to invent a new test split when the frozen manifest is missing.

In [ ]:
def find_repository_file(relative_path):
    relative_path = Path(relative_path)
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    return None

DATASET_PATH = find_repository_file('data/processed/survey-ml-cross-checkpoint-classification.csv')
SPLIT_PATH = find_repository_file('artifacts/splits/split_manifest.csv')

if DATASET_PATH is None or SPLIT_PATH is None:
    try:
        from google.colab import files
        print('Upload the processed CSV and split_manifest.csv to this private runtime.')
        uploaded = files.upload()
        for name in uploaded:
            path = Path('/content') / name
            if name == 'survey-ml-cross-checkpoint-classification.csv': DATASET_PATH = path
            if name == 'split_manifest.csv': SPLIT_PATH = path
    except ImportError:
        pass

if DATASET_PATH is None or not Path(DATASET_PATH).exists():
    raise FileNotFoundError('Processed dataset not found. Set DATASET_PATH without publishing the file.')
if SPLIT_PATH is None or not Path(SPLIT_PATH).exists():
    raise FileNotFoundError('Frozen split manifest not found. A replacement split is intentionally not generated.')

DATASET_PATH, SPLIT_PATH = Path(DATASET_PATH), Path(SPLIT_PATH)
print('Private inputs found. Participant-level contents will not be displayed.')

## 3. Fail-closed data validation

Only the checkpoint and 15 T0 questionnaire responses enter `X`. Follow-up checkpoints, follow-up scores, identifiers, profile context, and targets are prohibited from model inputs. The follow-up scores may verify target construction but never become predictors.

In [ ]:
baseline_oh = [f'baseline_checkpoint_{c.lower()}' for c in CHECKPOINTS]
followup_oh = [f'followup_checkpoint_{c.lower()}' for c in CHECKPOINTS]
profile_columns = ['profile_sex_male','profile_sex_female','profile_premature_yes','profile_premature_no',
    'profile_premature_not_sure','profile_gestational_weeks','profile_gestational_weeks_missing',
    'profile_birth_weight_under_1_5_kg','profile_birth_weight_1_5_to_1_99','profile_birth_weight_2_0_to_2_49',
    'profile_birth_weight_2_5_to_2_99','profile_birth_weight_3_0_to_3_49','profile_birth_weight_3_5_to_3_99',
    'profile_birth_weight_4_or_more','profile_birth_weight_not_sure']
score_columns = ['followup_communication_score','followup_cognitive_score','followup_social_emotional_score']
expected_columns = ['participant_number','reporting_method','baseline_assessment_id','baseline_checkpoint_id',
    'followup_checkpoint_id', *QUESTION_COLUMNS, *baseline_oh, *followup_oh, *profile_columns, *score_columns,
    *TARGET_COLUMNS, 'target_any_developmental_concern']

dataset_hash = sha256(DATASET_PATH.read_bytes()).hexdigest().upper()
manifest_hash = sha256(SPLIT_PATH.read_bytes()).hexdigest().upper()
assert dataset_hash == EXPECTED_DATASET_SHA256, 'Dataset checksum mismatch: stop and verify provenance.'
assert manifest_hash == EXPECTED_MANIFEST_SHA256, 'Split-manifest checksum mismatch: do not replace the locked split silently.'
df = pd.read_csv(DATASET_PATH)
assert df.shape == (625, 66), f'Expected 625 x 66, received {df.shape}.'
assert list(df.columns) == expected_columns, 'Source schema or column order changed.'
assert df['participant_number'].notna().all() and df['participant_number'].is_unique
assert df['baseline_assessment_id'].notna().all() and df['baseline_assessment_id'].is_unique
assert set(df['baseline_checkpoint_id']) <= set(CHECKPOINTS)
assert not df[QUESTION_COLUMNS].isna().any().any()
assert all(set(df[c].astype(int).unique()) <= {0,1,2} for c in QUESTION_COLUMNS)
assert all(set(df[c].astype(int).unique()) <= {0,1} for c in TARGET_COLUMNS)
for score, target in zip(score_columns, TARGET_COLUMNS):
    assert ((df[score].astype(float) <= 50).astype(int) == df[target].astype(int)).all(), f'{target} derivation mismatch'
assert (df[TARGET_COLUMNS].max(axis=1).astype(int) == df['target_any_developmental_concern'].astype(int)).all()
prohibited = set(df.columns) - set(FEATURE_COLUMNS)
assert not (set(FEATURE_COLUMNS) & prohibited)
print({'valid': True, 'rows': len(df), 'columns': len(df.columns), 'primary_features': len(FEATURE_COLUMNS),
       'dataset_sha256': dataset_hash, 'participant_rows_displayed': 0})

## 4. Privacy-safe aggregate EDA

The EDA below shows only counts and distributions. It intentionally does not call `head()`, print IDs, or display individual questionnaire patterns.

In [ ]:
target_summary = pd.DataFrame({
    'positive': df[TARGET_COLUMNS].sum().astype(int),
    'negative': (len(df) - df[TARGET_COLUMNS].sum()).astype(int),
    'positive_rate': df[TARGET_COLUMNS].mean()
})
display(target_summary.rename(index=dict(zip(TARGET_COLUMNS, DOMAIN_NAMES))).round(4))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
target_summary['positive_rate'].rename(index=dict(zip(TARGET_COLUMNS, DOMAIN_NAMES))).plot.bar(ax=axes[0], color='#4472C4')
axes[0].set(title='Follow-up concern prevalence', ylabel='Positive rate', ylim=(0, 0.25)); axes[0].tick_params(axis='x', rotation=20)
df['baseline_checkpoint_id'].value_counts().reindex(CHECKPOINTS).plot.bar(ax=axes[1], color='#70AD47')
axes[1].set(title='Baseline checkpoint distribution', ylabel='Participants'); axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
response_counts = pd.DataFrame({c: df[c].value_counts().reindex([0,1,2], fill_value=0) for c in QUESTION_COLUMNS}).T
response_counts.columns = ['NOT_YET (0)','SOMETIMES (1)','YES_USUALLY (2)']
response_counts.plot.bar(stacked=True, figsize=(15,5), color=['#C0504D','#F2C811','#70AD47'])
plt.title('Aggregate baseline response distributions'); plt.ylabel('Participants'); plt.xticks(rotation=60, ha='right'); plt.tight_layout(); plt.show()

## 5. Apply the frozen participant split

The 500-participant development partition is used for model comparison, calibration, and threshold selection. The 125-participant test partition is kept out of all those choices.

In [ ]:
manifest = pd.read_csv(SPLIT_PATH, dtype={'participant_number': str})
assert list(manifest.columns) == ['participant_number','partition']
assert len(manifest) == 625 and manifest['participant_number'].is_unique
assert set(manifest['partition']) == {'development','final_test'}
work = df.copy(); work['participant_number'] = work['participant_number'].astype(str)
work = work.merge(manifest, on='participant_number', how='left', validate='one_to_one')
assert work['partition'].notna().all()
development = work.loc[work['partition'].eq('development')].copy()
final_test = work.loc[work['partition'].eq('final_test')].copy()
assert len(development) == 500 and len(final_test) == 125
assert set(development['participant_number']).isdisjoint(set(final_test['participant_number']))
X_dev, Y_dev = development[FEATURE_COLUMNS], development[TARGET_COLUMNS].astype(int)
X_test, Y_test = final_test[FEATURE_COLUMNS], final_test[TARGET_COLUMNS].astype(int)
print({'development_participants': len(X_dev), 'final_test_participants': len(X_test),
       'participant_overlap': 0, 'development_positive_counts': Y_dev.sum().to_dict(),
       'test_positive_counts': Y_test.sum().to_dict()})

## 6. Fold-fitted preprocessing and candidate models

`baseline_checkpoint_id` is encoded categorically inside every fitted pipeline. The 15 responses remain ordered numeric values. Scaling is learned inside each training fold for Logistic Regression and SVM. Tree models do not require scaling. Class balancing is also learned/applied using training-fold data only.

In [ ]:
@dataclass(frozen=True)
class Candidate:
    name: str
    estimator: object
    scale_questions: bool
    balanced_sample_weight: bool = False

CANDIDATES = [
 Candidate('logistic_l2_balanced', LogisticRegression(C=1.0, penalty='l2', class_weight='balanced', solver='liblinear', max_iter=2000, random_state=SEED), True),
 Candidate('svm_rbf_balanced', SVC(C=1.0, kernel='rbf', gamma='scale', class_weight='balanced', probability=True, random_state=SEED), True),
 Candidate('random_forest_balanced_leaf5', RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', random_state=SEED, n_jobs=1), False),
 Candidate('extra_trees_balanced_leaf5', ExtraTreesClassifier(n_estimators=300, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', random_state=SEED, n_jobs=1), False),
 Candidate('xgboost_regularized_balanced', XGBClassifier(objective='binary:logistic', eval_metric='logloss', tree_method='hist', n_estimators=300, learning_rate=.03, max_depth=3, min_child_weight=5, subsample=.8, colsample_bytree=.8, reg_alpha=.1, reg_lambda=5, random_state=SEED, n_jobs=1), False, True),
]

def build_pipeline(candidate):
    preprocessor = ColumnTransformer([
        ('checkpoint', OneHotEncoder(categories=[CHECKPOINTS], drop='first', handle_unknown='ignore', sparse_output=False), ['baseline_checkpoint_id']),
        ('questions', StandardScaler() if candidate.scale_questions else 'passthrough', QUESTION_COLUMNS),
    ], remainder='drop', verbose_feature_names_out=False)
    return Pipeline([('preprocess', preprocessor), ('estimator', clone(candidate.estimator))])

def fit_candidate(candidate, X, y):
    pipeline = build_pipeline(candidate)
    params = {}
    if candidate.balanced_sample_weight:
        params['estimator__sample_weight'] = compute_sample_weight('balanced', y)
    return pipeline.fit(X, y, **params)

print('Candidates:', [c.name for c in CANDIDATES])

## 7. Development-only repeated cross-validation

All candidates and domains use the same five-fold, five-repeat allocation. Stratification uses the combined three-label pattern only to allocate folds; targets never enter the model features. PR-AUC is the primary ranking summary because positives are uncommon.

In [ ]:
def metric_row(y, p, threshold=0.5):
    d = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, d, labels=[0,1]).ravel()
    return {'sensitivity': recall_score(y,d,zero_division=0), 'specificity': tn/(tn+fp) if tn+fp else 0,
        'precision': precision_score(y,d,zero_division=0), 'f1': f1_score(y,d,zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y,d), 'accuracy': accuracy_score(y,d),
        'pr_auc': average_precision_score(y,p), 'roc_auc': roc_auc_score(y,p),
        'brier': brier_score_loss(y,p), 'tn':int(tn), 'fp':int(fp), 'fn':int(fn), 'tp':int(tp)}

labels = Y_dev.astype(str).agg(''.join, axis=1)
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=20260821)
folds = list(cv.split(X_dev, labels))
oof = {}
comparison_rows = []
for candidate in CANDIDATES:
    oof[candidate.name] = {}
    for domain_index, target in enumerate(TARGET_COLUMNS):
        probabilities = np.full((N_REPEATS, len(X_dev)), np.nan)
        for fold_number, (train_idx, valid_idx) in enumerate(folds):
            repeat = fold_number // N_SPLITS
            fitted = fit_candidate(candidate, X_dev.iloc[train_idx], Y_dev[target].iloc[train_idx])
            probabilities[repeat, valid_idx] = fitted.predict_proba(X_dev.iloc[valid_idx])[:,1]
        assert np.isfinite(probabilities).all()
        oof[candidate.name][target] = probabilities
        averaged = probabilities.mean(axis=0)
        comparison_rows.append({'candidate': candidate.name, 'domain': DOMAIN_NAMES[domain_index], **metric_row(Y_dev[target], averaged)})
        print(f'finished: {candidate.name} / {DOMAIN_NAMES[domain_index]}')
comparison = pd.DataFrame(comparison_rows)
display(comparison[['candidate','domain','pr_auc','sensitivity','specificity','precision','balanced_accuracy','brier']].sort_values(['domain','pr_auc'], ascending=[True,False]).round(4))

## 8. Select one candidate per domain, calibrate, and freeze thresholds

Selection uses development PR-AUC only. A fold-safe sigmoid calibrator is assessed on development OOF predictions. Each threshold maximizes balanced accuracy (equivalent to Youden's J), with deterministic ties preferring sensitivity, then specificity, then the lower threshold. These newly derived thresholds are experimental and are not automatically clinically approved.

In [ ]:
def logit_feature(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1-1e-6)
    return np.log(p/(1-p)).reshape(-1,1)

def select_threshold(y, p):
    best = None
    for threshold in np.unique(np.clip(p, 1e-6, 1-1e-6)):
        m = metric_row(y, p, float(threshold))
        key = (m['balanced_accuracy'], m['sensitivity'], m['specificity'], -float(threshold))
        if best is None or key > best[0]: best = (key, float(threshold), m)
    return best[1], best[2]

selected_names, calibrated_oof, final_calibrators, thresholds = {}, {}, {}, {}
for target, domain in zip(TARGET_COLUMNS, DOMAIN_NAMES):
    rows = comparison.loc[comparison['domain'].eq(domain)].sort_values(['pr_auc','balanced_accuracy'], ascending=False)
    selected_names[target] = rows.iloc[0]['candidate']
    raw = oof[selected_names[target]][target].mean(axis=0)
    y = Y_dev[target].to_numpy()
    calibrated = np.full(len(y), np.nan)
    calibration_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for train_idx, valid_idx in calibration_cv.split(raw, y):
        cal = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000, random_state=SEED)
        cal.fit(logit_feature(raw[train_idx]), y[train_idx])
        calibrated[valid_idx] = cal.predict_proba(logit_feature(raw[valid_idx]))[:,1]
    final_cal = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000, random_state=SEED)
    final_cal.fit(logit_feature(raw), y)
    threshold, threshold_metrics = select_threshold(y, calibrated)
    calibrated_oof[target], final_calibrators[target], thresholds[target] = calibrated, final_cal, threshold
    print(domain, {'selected': selected_names[target], 'threshold': round(threshold,6),
                   'development_pr_auc': round(average_precision_score(y,calibrated),4),
                   'development_sensitivity': round(threshold_metrics['sensitivity'],4),
                   'development_specificity': round(threshold_metrics['specificity'],4)})

## 9. Fit frozen models and evaluate the locked test once

No test result is used to change candidate choice, calibration, or thresholds. Confidence intervals are participant-level bootstrap intervals and will be wide because each test-domain has only about 14–15 positives.

In [ ]:
candidate_lookup = {c.name:c for c in CANDIDATES}
final_models, test_probabilities, test_results = {}, {}, []
for target, domain in zip(TARGET_COLUMNS, DOMAIN_NAMES):
    model = fit_candidate(candidate_lookup[selected_names[target]], X_dev, Y_dev[target])
    raw_test = model.predict_proba(X_test)[:,1]
    calibrated_test = final_calibrators[target].predict_proba(logit_feature(raw_test))[:,1]
    final_models[target], test_probabilities[target] = model, calibrated_test
    test_results.append({'domain':domain, 'candidate':selected_names[target], 'threshold':thresholds[target],
                         **metric_row(Y_test[target], calibrated_test, thresholds[target])})
test_results = pd.DataFrame(test_results)
display(test_results[['domain','candidate','threshold','pr_auc','sensitivity','specificity','precision','f1','balanced_accuracy','brier','tn','fp','fn','tp']].round(4))

predicted_domains = np.column_stack([(test_probabilities[t] >= thresholds[t]).astype(int) for t in TARGET_COLUMNS])
true_any = Y_test.max(axis=1).to_numpy()
predicted_any = predicted_domains.max(axis=1)
print('Derived any-concern check (OR; no fourth model):', metric_row(true_any, predicted_any.astype(float), 0.5))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13,4))
for ax, target, domain in zip(axes, TARGET_COLUMNS, DOMAIN_NAMES):
    decision = (test_probabilities[target] >= thresholds[target]).astype(int)
    cm = confusion_matrix(Y_test[target], decision, labels=[0,1])
    ax.imshow(cm, cmap='Blues')
    for (i,j), value in np.ndenumerate(cm): ax.text(j,i,str(value),ha='center',va='center')
    ax.set(title=domain, xlabel='Predicted', ylabel='Actual', xticks=[0,1], yticks=[0,1])
plt.suptitle('Locked-test confusion matrices'); plt.tight_layout(); plt.show()

def bootstrap_intervals(y, p, threshold, repeats=2000, seed=SEED):
    rng = np.random.default_rng(seed); rows=[]; y=np.asarray(y); p=np.asarray(p)
    for _ in range(repeats):
        idx=rng.integers(0,len(y),len(y))
        if np.unique(y[idx]).size < 2: continue
        rows.append(metric_row(y[idx],p[idx],threshold))
    return {m:(np.quantile([r[m] for r in rows],.025),np.quantile([r[m] for r in rows],.975))
            for m in ['sensitivity','specificity','precision','balanced_accuracy','pr_auc','brier']}
ci_rows=[]
for target, domain in zip(TARGET_COLUMNS, DOMAIN_NAMES):
    intervals=bootstrap_intervals(Y_test[target],test_probabilities[target],thresholds[target])
    for metric,(low,high) in intervals.items(): ci_rows.append({'domain':domain,'metric':metric,'lower_95':low,'upper_95':high})
display(pd.DataFrame(ci_rows).round(4))

## 10. Aggregate explainability

Held-out permutation importance measures predictive association, not causation. Negative or unstable values are possible with this small test set. Technical values are for research/audit views and must not be shown directly to parents.

In [ ]:
importance_rows=[]
for target, domain in zip(TARGET_COLUMNS, DOMAIN_NAMES):
    result=permutation_importance(final_models[target],X_test,Y_test[target],scoring='average_precision',n_repeats=20,random_state=SEED,n_jobs=1)
    for feature,mean,std in zip(FEATURE_COLUMNS,result.importances_mean,result.importances_std):
        importance_rows.append({'domain':domain,'feature':feature,'mean_importance':mean,'std':std})
importance=pd.DataFrame(importance_rows)
for domain in DOMAIN_NAMES:
    display(importance.loc[importance.domain.eq(domain)].nlargest(10,'mean_importance').round(4))

## 11. Export an immutable inference bundle

The bundle contains three fitted pipelines, three calibrators, thresholds, and schema/version metadata. It contains no CSV rows or participant identifiers. Treat it as a research artifact, not a clinically approved model. Production must verify its checksum and keep training code separate from the inference service.

In [ ]:
IN_COLAB = Path('/content').exists() and 'google.colab' in sys.modules
OUTPUT_DIR = Path('/content/slc_model_artifacts') if IN_COLAB else Path('tmp/full-training-pipeline-export')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
bundle = {'artifact_schema_version':'colab-end-to-end-research-v1', 'research_only':True,
    'feature_schema_version':'primary-t0-16-v1', 'target_definition_version':'followup-domain-score-le-50-v1',
    'feature_columns':FEATURE_COLUMNS, 'target_columns':TARGET_COLUMNS, 'checkpoint_categories':CHECKPOINTS,
    'dataset_sha256':dataset_hash, 'split_manifest_sha256':manifest_hash,
    'selected_candidates':selected_names, 'thresholds':thresholds, 'models':final_models, 'calibrators':final_calibrators}
bundle_path=OUTPUT_DIR/'slc_three_domain_model_bundle.joblib'
joblib.dump(bundle,bundle_path)
bundle_hash=sha256(bundle_path.read_bytes()).hexdigest().upper()
manifest_out={'artifact':bundle_path.name,'sha256':bundle_hash,'research_only':True,
    'contains_participant_identifiers':False,'feature_schema_version':'primary-t0-16-v1',
    'target_definition_version':'followup-domain-score-le-50-v1','selected_candidates':selected_names,
    'thresholds':thresholds,'library_versions':{'numpy':np.__version__,'pandas':pd.__version__,
    'scikit-learn':version('scikit-learn'),'xgboost':version('xgboost'),'joblib':version('joblib')}}
manifest_out_path=OUTPUT_DIR/'manifest.json'
manifest_out_path.write_text(json.dumps(manifest_out,indent=2),encoding='utf-8')
zip_path=OUTPUT_DIR.parent/'slc_model_artifacts.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
    archive.write(bundle_path,bundle_path.name); archive.write(manifest_out_path,manifest_out_path.name)
print({'bundle':str(bundle_path),'bundle_sha256':bundle_hash,'archive':str(zip_path),'identifiers_exported':False})

## 12. Production-contract demonstration

Every field is required. `NOT_SURE` and `NOT_OBSERVED` are valid but unscorable and result in `INCOMPLETE`; they are never converted to zero. Only complete assessments reach ML inference. The parent-facing result omits probabilities and diagnostic language. The example below is synthetic and not a participant record.

In [ ]:
UNSCORABLE={'NOT_SURE','NOT_OBSERVED'}
def predict_assessment(submission, loaded_bundle=bundle):
    required=set(FEATURE_COLUMNS)
    missing=required-set(submission); unexpected=set(submission)-required
    if missing or unexpected: raise ValueError(f'Exact schema required; missing={sorted(missing)}, unexpected={sorted(unexpected)}')
    if submission['baseline_checkpoint_id'] not in CHECKPOINTS: raise ValueError('Unknown baseline checkpoint.')
    answers=[submission[c] for c in QUESTION_COLUMNS]
    if any(v in UNSCORABLE for v in answers):
        return {'status':'INCOMPLETE','domain_decisions':None,'overall_concern':None,
            'parent_message':'Some items need observation before a developmental result can be provided.'}
    if any(v not in {0,1,2} for v in answers): raise ValueError('Responses must be 0, 1, 2, NOT_SURE, or NOT_OBSERVED.')
    frame=pd.DataFrame([{c:submission[c] for c in FEATURE_COLUMNS}],columns=FEATURE_COLUMNS)
    technical_probabilities={}; decisions={}
    for target in TARGET_COLUMNS:
        raw=loaded_bundle['models'][target].predict_proba(frame)[:,1]
        probability=float(loaded_bundle['calibrators'][target].predict_proba(logit_feature(raw))[:,1][0])
        technical_probabilities[target]=probability
        decisions[target]=bool(probability >= loaded_bundle['thresholds'][target])
    return {'status':'COMPLETE','domain_decisions':decisions,'overall_concern':any(decisions.values()),
        'technical_probabilities_for_research_audit_only':technical_probabilities,
        'parent_message':'One or more developmental areas may benefit from continued observation and age-appropriate activities.' if any(decisions.values()) else 'Continue age-appropriate activities and routine developmental monitoring.'}

synthetic_example={'baseline_checkpoint_id':'CP24M', **{c:1 for c in QUESTION_COLUMNS}}
example_result=predict_assessment(synthetic_example)
display({k:v for k,v in example_result.items() if k != 'technical_probabilities_for_research_audit_only'})

## 13. TabICLv2 and the current experimental prototype

The repository's consolidated prototype retains **XGBoost for Communication and Social-emotional** and an experimental **TabICLv2 model for Cognitive**. TabICLv2 is not silently run here because exact reproduction requires `tabicl==2.2.0` and the separately pinned checkpoint `tabicl-classifier-v2-20260212.ckpt` with its verified checksum. Automatically downloading or substituting that checkpoint would weaken provenance.

Use the repository's `training/tabiclv2/candidate.py` and staged TabICLv2 experiment scripts when the verified checkpoint is available. Its later result on the already-used 125-participant cohort is secondary exploratory evidence, not fresh validation. The bundle created above is therefore a new transparent standard-candidate research run, not a claim that it reproduces the existing TabICLv2 prototype.

## Final interpretation checklist

- Report Communication, Cognitive, and Social-emotional results separately.
- Prioritize sensitivity constraints approved by appropriate reviewers; also report specificity and precision.
- Use PR-AUC for imbalanced ranking comparisons and Brier/reliability evidence for probabilities.
- Show confusion-matrix counts and confidence intervals; accuracy alone is misleading.
- Overall concern is the OR of three decisions, never a fourth primary model.
- ML, deterministic safety/referral rules, approved activity selection, and optional LLM wording remain separate boundaries.
- Do not show raw probabilities or technical explanation values to parents.

# 14. Complete staged research reproduction: eleven families and hybrids

The readable pipeline above reproduces the foundational five-family experiment in one place. The project subsequently assessed six more families in controlled stages. They were **not originally one simultaneous eleven-model experiment**, so this section preserves that evidence history and invokes the repository's exact, tested experiment implementations rather than rewriting them approximately.

| Stage | Algorithm families or methods | Evidence boundary |
|---|---|---|
| Foundational | Logistic Regression, RBF SVM, Random Forest, Extra Trees, XGBoost | Shared development folds |
| Additional | CatBoost, Explainable Boosting Machine | Nested configuration selection, calibration, thresholds |
| Later fixed candidates | HistGradientBoosting, LightGBM, FIGS | Separate development-only stages using the same frozen contract |
| Foundation-model candidate | TabICLv2 | Requires the verified external checkpoint; development-only primary comparison |
| Hybrid A | Logistic Regression + XGBoost weighted soft voting | Nested development-only weight selection |
| Hybrid B | Cognitive TabICLv2 + Logistic Regression | Post-hoc exploratory development-only analysis |

All stages retain the same 16 T0 inputs and three separate targets. No later model may use the already-evaluated locked test to choose a model, weight, calibration method, or threshold.

## 14.1 Prepare the exact repository implementation

In Colab this cell clones the public source code, then places the already checksum-verified private dataset and manifest into the private runtime checkout. It creates only the aggregate split summary required by the fail-closed loader; it does not print or persist participant rows in notebook outputs.

The optional model packages are isolated in the repository design because they have heavier and sometimes conflicting dependencies. Colab uses one temporary runtime, so restart the runtime after installation if Colab requests it.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/JayasankaHirimuthugodage/ResearchComponent.git'
def locate_repository_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base/'training/contracts.py').exists() and (base/'pyproject.toml').exists(): return base
    destination = Path('/content/ResearchComponent')
    if not destination.exists():
        subprocess.run(['git','clone','--depth','1',REPOSITORY_URL,str(destination)],check=True)
    return destination

RESEARCH_ROOT = locate_repository_root().resolve()
private_data_destination = RESEARCH_ROOT/'data/processed/survey-ml-cross-checkpoint-classification.csv'
private_manifest_destination = RESEARCH_ROOT/'artifacts/splits/split_manifest.csv'
private_data_destination.parent.mkdir(parents=True,exist_ok=True)
private_manifest_destination.parent.mkdir(parents=True,exist_ok=True)
if DATASET_PATH.resolve() != private_data_destination.resolve(): shutil.copy2(DATASET_PATH,private_data_destination)
if SPLIT_PATH.resolve() != private_manifest_destination.resolve(): shutil.copy2(SPLIT_PATH,private_manifest_destination)

manifest_runtime=pd.read_csv(private_manifest_destination,dtype={'participant_number':str})
fingerprint_source='\n'.join(f"{row.participant_number}:{row.partition}" for row in manifest_runtime.itertuples(index=False)).encode()
split_fingerprint=sha256(fingerprint_source).hexdigest().upper()
assert split_fingerprint == 'BD6D8EE8BEA1D8893E2DC041F7FFF49642E70FD93019FB0DE30A9042626392F0'
split_summary={'dataset_sha256':dataset_hash,'manifest_sha256':manifest_hash,'split_fingerprint_sha256':split_fingerprint,
               'development':{'participants':int((manifest_runtime.partition=='development').sum())},
               'final_test':{'participants':int((manifest_runtime.partition=='final_test').sum())}}
split_summary_path=RESEARCH_ROOT/'artifacts/splits/split_summary.json'
if split_summary_path.exists():
    existing_summary=json.loads(split_summary_path.read_text(encoding='utf-8'))
    for key in ['dataset_sha256','manifest_sha256','split_fingerprint_sha256']:
        assert existing_summary.get(key)==split_summary[key], f'Existing split summary mismatch: {key}'
else:
    split_summary_path.write_text(json.dumps(split_summary,indent=2),encoding='utf-8')
print({'repository_ready':True,'dataset_checksum_verified':True,'split_fingerprint_verified':True,
       'participant_rows_displayed':0})

## 14.2 Install optional algorithm families

CatBoost and EBM reproduce the additional-model stage. LightGBM and FIGS reproduce later fixed-complexity stages. TabICLv2 is installed only when its verified checkpoint is supplied because the project prohibits silent model-weight downloads.

In [ ]:
OPTIONAL_PACKAGES=['catboost==1.2.8','interpret-core==0.7.8','lightgbm==4.6.0','imodels==3.0.4']
subprocess.run([sys.executable,'-m','pip','install','-q',*OPTIONAL_PACKAGES],check=True)

TABICLV2_CHECKPOINT_SHA256='BDC7DBD5E4FF21F8F0456FCF90C6B7CDF72DBEA960F2D05B19BEC19F9B3D4ED0'
TABICLV2_CHECKPOINT_NAME='tabicl-classifier-v2-20260212.ckpt'
TABICLV2_CHECKPOINT_PATH=Path('/content')/TABICLV2_CHECKPOINT_NAME
RUN_TABICLV2=TABICLV2_CHECKPOINT_PATH.exists()
if RUN_TABICLV2:
    assert sha256(TABICLV2_CHECKPOINT_PATH.read_bytes()).hexdigest().upper()==TABICLV2_CHECKPOINT_SHA256, 'TabICLv2 checkpoint checksum mismatch'
    subprocess.run([sys.executable,'-m','pip','install','-q','tabicl==2.2.0'],check=True)
else:
    print(f'TabICLv2 blocked: privately upload {TABICLV2_CHECKPOINT_NAME}; automatic download is disabled.')
print({'optional_standard_families_ready':True,'tabiclv2_ready':RUN_TABICLV2})

## 14.3 Execute the development-only stages

This cell uses the exact repository runners. Completed aggregate reports are reused rather than overwritten. On a fresh Colab checkout, the full five-repeat run can take a long time, especially CatBoost, EBM, FIGS, and TabICLv2. The final-test evaluator is deliberately absent: the cohort has already been evaluated and is not a reusable model-selection resource.

In [ ]:
RUN_FULL_STAGED_DEVELOPMENT=True
generated=RESEARCH_ROOT/'reports/generated'; generated.mkdir(parents=True,exist_ok=True)
def run_stage(module,report_name,extra_args=()):
    report=generated/report_name
    if report.exists():
        print('reuse aggregate report:',report_name); return 'reused'
    command=[sys.executable,'-B','-m',module,*map(str,extra_args)]
    subprocess.run(command,cwd=RESEARCH_ROOT,check=True)
    if not report.exists(): raise RuntimeError(f'{module} completed without {report_name}')
    return 'completed'

stage_status={}
if RUN_FULL_STAGED_DEVELOPMENT:
    stage_status['five_foundational']=run_stage('training.experiments.run_development_experiment','development-model-comparison-report.json')
    stage_status['calibration']=run_stage('training.experiments.run_calibration_experiment','development-calibration-report.json')
    stage_status['decision']=run_stage('training.experiments.generate_development_decision_report','development-decision-report.json')
    stage_status['catboost_ebm']=run_stage('training.experiments.run_additional_model_experiment','development-additional-model-comparison-report.json')
    stage_status['histgradientboosting']=run_stage('training.experiments.run_histgradientboosting_development','development-histgradientboosting-comparison-report.json')
    stage_status['lightgbm']=run_stage('training.experiments.run_lightgbm_development','development-lightgbm-comparison-report.json')
    stage_status['logistic_xgboost_hybrid']=run_stage('training.experiments.run_hybrid_development_experiment','development-hybrid-model-comparison-report.json')
    if RUN_TABICLV2:
        stage_status['tabiclv2']=run_stage('training.experiments.run_tabiclv2_development_experiment','development-tabiclv2-model-comparison-report.json',('--checkpoint',TABICLV2_CHECKPOINT_PATH))
        stage_status['figs']=run_stage('training.experiments.run_figs_development','development-figs-comparison-report.json')
        stage_status['cognitive_tabiclv2_finalization']=run_stage('training.experiments.run_cognitive_tabiclv2_finalization','development-cognitive-tabiclv2-finalization-report.json',('--checkpoint',TABICLV2_CHECKPOINT_PATH,'--execute'))
        stage_status['cognitive_tabiclv2_logistic_hybrid']=run_stage('training.experiments.run_cognitive_tabiclv2_weighted_hybrid','development-cognitive-tabiclv2-weighted-hybrid-report.json',('--checkpoint',TABICLV2_CHECKPOINT_PATH))
    else:
        stage_status.update({'tabiclv2':'blocked_missing_verified_checkpoint','figs':'blocked_until_tabiclv2_report',
                             'cognitive_tabiclv2_finalization':'blocked_missing_verified_checkpoint',
                             'cognitive_tabiclv2_logistic_hybrid':'blocked_missing_verified_checkpoint'})
display(pd.DataFrame([{'stage':k,'status':v} for k,v in stage_status.items()]))

## 14.4 Consolidate the eleven-family evidence

This summary verifies which families actually produced development-only reports. It does not merge incomparable stages into a false single leaderboard. Model-family conclusions must retain the protocol/source column, uncertainty, and the fact that later candidates were exploratory.

In [ ]:
family_sources={
 'Logistic Regression':'development-model-comparison-report.json','RBF SVM':'development-model-comparison-report.json',
 'Random Forest':'development-model-comparison-report.json','Extra Trees':'development-model-comparison-report.json',
 'XGBoost':'development-model-comparison-report.json','CatBoost':'development-additional-model-comparison-report.json',
 'Explainable Boosting Machine':'development-additional-model-comparison-report.json',
 'HistGradientBoosting':'development-histgradientboosting-comparison-report.json',
 'LightGBM':'development-lightgbm-comparison-report.json','FIGS':'development-figs-comparison-report.json',
 'TabICLv2':'development-tabiclv2-model-comparison-report.json'}
family_evidence=pd.DataFrame([{'algorithm_family':name,'source_report':report,
    'report_available':(generated/report).exists(),'scope':'development only'} for name,report in family_sources.items()])
display(family_evidence)
assert family_evidence.algorithm_family.nunique()==11

hybrid_sources={'Logistic + XGBoost':'development-hybrid-model-comparison-report.json',
                'Cognitive TabICLv2 + Logistic':'development-cognitive-tabiclv2-weighted-hybrid-report.json'}
display(pd.DataFrame([{'hybrid':name,'source_report':report,'report_available':(generated/report).exists(),
    'status':'exploratory; requires new untouched/external validation'} for name,report in hybrid_sources.items()]))

## 14.5 What is exported and what is not

The earlier export cell creates a portable bundle for the self-contained standard-candidate rerun. It is not silently relabelled as the historical prototype. The staged runners produce checksum-linked aggregate reports and, for Cognitive TabICLv2 finalization, a development-only release manifest. TabICLv2 reconstructs its 500-person development context from authorized checksum-verified data in memory; embedding that context in a portable public notebook or model archive would violate the privacy boundary.

Accordingly, the current experimental architecture remains:

- Communication: XGBoost
- Cognitive: TabICLv2, experimental and not deployed
- Social-emotional: XGBoost
- Overall: logical OR of the three domain decisions

The hybrid studies remain exploratory because their incremental PR-AUC improvements were small or uncertain and did not justify replacing the simpler domain-specific configuration. A new untouched or external cohort is required before promoting a later candidate or hybrid.